# Impact of LS cocktail on TDCR detection efficiency

Compares **12 commercial LS cocktails** available in TDCRPy for three nuclides:

| Nuclide | Decay | Key physics |
|---------|-------|-------------|
| **H-3** | Pure β⁻, 18.6 keV | Low-energy quenching |
| **C-14** | Pure β⁻, 156 keV | Medium-energy quenching |
| **Fe-55** | EC + Mn Kα X-rays, 5.9 keV | Photoelectric interaction (sensitive to heavy elements P, S, Na, Cl) |

All runs use the **stochastic model** (N = 2000) for consistency across nuclide types.
Reference cocktail: **Ultima Gold** (solid bars; all residuals relative to it).

In [ ]:
pip install TDCRPy==2.20.12

In [ ]:
import importlib, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tdcrpy.TDCR_model_lib as tl
import tdcrpy.TDCRPy as tm
from importlib.metadata import version
warnings.filterwarnings('ignore')
print(f'TDCRPy {version("TDCRPy")}')

## ⚙️ Configuration

In [ ]:
L   = 5.0       # free parameter (photons keV⁻¹)
kB  = 1.0e-5    # Birks constant (cm keV⁻¹)
V   = 10        # scintillator volume (mL)
N   = 2000      # Monte-Carlo trials per run

NUCLIDES = ['H-3', 'C-14', 'Fe-55']
NUC_COL  = ['#1f77b4', '#2ca02c', '#e67e22']
REF      = 'Ultima Gold'

# Commercial cocktails to compare (pure cocktail, no aqueous fraction)
COCKTAILS = [
    'Ultima Gold',
    'Ultima Gold XR',
    'Ultima Gold AB',
    'Ultima Gold LLT',
    'Insta-Gel Plus',
    'Hionic-Fluor',
    'ProSafe+',
    'ProSafe HC+',
    'Pico-Fluor 40',
    'Pico-Fluor Plus',
    'Aqualight Beta',
    'Permafluor E+',
]

print(f'L={L} photons/keV  kB={kB} cm/keV  V={V} mL  N={N}')
print(f'{len(COCKTAILS)} cocktails × {len(NUCLIDES)} nuclides = {len(COCKTAILS)*len(NUCLIDES)} runs')

## 1. Cocktail properties

In [ ]:
props = {}
print(f'{"Cocktail":<30} {"rho":>6}  {"Z":>6}  {"A":>6}  {"Z/A":>6}  {"P+S+Na+Cl (%)"}' )
print('─' * 80)
for name in COCKTAILS:
    tl.modifyLScocktail(name, 0.0, 'Water', 0.0)
    tl = importlib.reload(tl)
    Z = tl.Z; A = tl.A; rho = tl.RHO
    heavy = (tl.pP + tl.pS + tl.pNa + tl.pCl) * 100  # atomic % heavy elements
    props[name] = {'Z': Z, 'A': A, 'rho': rho, 'heavy': heavy}
    print(f'{name:<30} {rho:6.4f}  {Z:6.4f}  {A:6.4f}  {Z/A:6.4f}  {heavy:6.3f}')

# Restore
tl.modifyLScocktail('Ultima Gold', 0.0, 'Water', 0.0)
tl = importlib.reload(tl)

## 2. Efficiency scan — stochastic model

In [ ]:
# results[cocktail][nuclide] = {'eff_S':, 'u_S':, 'eff_D':, 'u_D':, 'eff_T':, 'u_T':}
results = {c: {} for c in COCKTAILS}

for name in COCKTAILS:
    tl.modifyLScocktail(name, 0.0, 'Water', 0.0)
    tl = importlib.reload(tl)
    import tdcrpy.TDCRPy as tm   # reload TDCRPy to pick up fresh config
    tm = importlib.reload(tm)

    print(f'\n{name}')
    for rad in NUCLIDES:
        r = tm.TDCRPy(L, rad, '1', N, kB, V)
        results[name][rad] = {
            'eff_S': r[0], 'u_S': r[1],
            'eff_D': r[2], 'u_D': r[3],
            'eff_T': r[4], 'u_T': r[5],
        }
        print(f'  {rad:<6}  eff_D={r[2]:.5f}±{r[3]:.5f}  eff_T={r[4]:.5f}±{r[5]:.5f}')

# Restore defaults
tl.modifyLScocktail('Ultima Gold', 0.0, 'Water', 0.0)
tl = importlib.reload(tl)
print('\nDone.')

## 3. Efficiency comparison — split by nuclide

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 13))
fig.suptitle(f'LS cocktail effect on detection efficiency\nL={L} photons/keV  kB={kB} cm/keV  N={N}',
             fontsize=12)

n_c = len(COCKTAILS)
x   = np.arange(n_c)
w   = 0.35
ref_idx = COCKTAILS.index(REF)

for row, (rad, col) in enumerate(zip(NUCLIDES, NUC_COL)):
    eff_D = np.array([results[c][rad]['eff_D'] for c in COCKTAILS])
    u_D   = np.array([results[c][rad]['u_D']   for c in COCKTAILS])
    eff_T = np.array([results[c][rad]['eff_T'] for c in COCKTAILS])
    u_T   = np.array([results[c][rad]['u_T']   for c in COCKTAILS])

    # Sort by eff_D descending
    order  = np.argsort(eff_D)[::-1]
    c_sort = [COCKTAILS[i] for i in order]
    dD_s = eff_D[order]; uD_s = u_D[order]
    dT_s = eff_T[order]; uT_s = u_T[order]
    ref_pos = c_sort.index(REF)

    # Panel left: eff_D and eff_T bars
    ax = axes[row, 0]
    bars_D = ax.bar(x - w/2, dD_s, w, yerr=uD_s, capsize=3,
                    color=col, alpha=0.85, label='eff_D')
    bars_T = ax.bar(x + w/2, dT_s, w, yerr=uT_s, capsize=3,
                    color=col, alpha=0.4, hatch='//', label='eff_T')
    # Highlight reference
    bars_D[ref_pos].set_edgecolor('black'); bars_D[ref_pos].set_linewidth(2)
    bars_T[ref_pos].set_edgecolor('black'); bars_T[ref_pos].set_linewidth(2)
    ax.set_xticks(x)
    ax.set_xticklabels(c_sort, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Detection efficiency', fontsize=10)
    ax.set_title(f'{rad} — sorted by eff_D ↓', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    ax.text(ref_pos, ax.get_ylim()[0], '▲', ha='center', va='bottom',
            color='black', fontsize=10, fontweight='bold')

    # Panel right: residual vs REF (×10³)
    ax = axes[row, 1]
    ref_D = results[REF][rad]['eff_D']
    resid = (eff_D - ref_D) * 1e3   # ×10³
    ures  = u_D * 1e3
    colors_r = [col if i != ref_idx else 'gray' for i in range(n_c)]
    ax.axhline(0, color='black', lw=1)
    ax.bar(x, resid, color=colors_r, alpha=0.75, yerr=ures, capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels(COCKTAILS, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(f'Δeff_D (×10³) vs {REF}', fontsize=10)
    ax.set_title(f'{rad} — residual', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('cocktailResponse_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 4. Cocktail physical properties

In [ ]:
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4))
fig2.suptitle('Scintillator physical properties', fontsize=11)

ZA   = [props[c]['Z'] / props[c]['A']   for c in COCKTAILS]
rhos = [props[c]['rho']                  for c in COCKTAILS]
hvy  = [props[c]['heavy']               for c in COCKTAILS]

for ax, vals, ylabel, title, color in [
    (axes2[0], ZA,   'Z/A',                   'Effective Z/A',            'purple'),
    (axes2[1], rhos, 'Density (g/cm³)',        'Scintillator density',     'brown'),
    (axes2[2], hvy,  'P+S+Na+Cl atomic (%)',   'Heavy-element content',    'red'),
]:
    ref_val = vals[COCKTAILS.index(REF)]
    bar_c = [color if v != ref_val else 'gray' for v in vals]
    ax.bar(np.arange(len(COCKTAILS)), vals, color=bar_c, alpha=0.75)
    ax.axhline(ref_val, color='black', lw=1, ls='--')
    ax.set_xticks(np.arange(len(COCKTAILS)))
    ax.set_xticklabels(COCKTAILS, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('cocktailResponse_properties.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 5. Correlation: efficiency vs cocktail properties

For pure β emitters (H-3, C-14) quenching depends on Z/A and density.
For Fe-55 (EC + X-ray) the photoelectric cross-section at 5.9 keV
is higher for heavy elements (P, S, Na, Cl with Z > 8), so cocktails
rich in these elements may show a different response.

In [ ]:
fig3, axes3 = plt.subplots(2, 3, figsize=(15, 9))
fig3.suptitle('Efficiency correlations with cocktail composition', fontsize=11)

x_vars = [
    ('Z/A',        [props[c]['Z']/props[c]['A'] for c in COCKTAILS], 'Effective Z/A'),
    ('density',    [props[c]['rho']             for c in COCKTAILS], 'Density (g/cm³)'),
    ('heavy',      [props[c]['heavy']           for c in COCKTAILS], 'P+S+Na+Cl atomic (%)'),
]

for col_ax, (xkey, xvals, xlabel) in enumerate(x_vars):
    for row_ax, (rad, col) in enumerate([(('H-3','C-14'), NUC_COL[:2]), (('Fe-55',), [NUC_COL[2]])]):
        ax = axes3[row_ax, col_ax]
        for r, c in zip(rad, col):
            yvals = [results[name][r]['eff_D'] for name in COCKTAILS]
            yerrs = [results[name][r]['u_D']   for name in COCKTAILS]
            ax.errorbar(xvals, yvals, yerr=yerrs, fmt='o', color=c, capsize=3,
                        markersize=6, label=r)
            # Label the reference
            idx_ref = COCKTAILS.index(REF)
            ax.annotate(REF, (xvals[idx_ref], yvals[idx_ref]),
                        textcoords='offset points', xytext=(4, 4), fontsize=7, color=c)
            # Label Hionic-Fluor (high P)
            idx_h = COCKTAILS.index('Hionic-Fluor')
            ax.annotate('Hionic-Fluor', (xvals[idx_h], yvals[idx_h]),
                        textcoords='offset points', xytext=(4, -10), fontsize=7, color=c)
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel('eff_D', fontsize=10)
        ax.set_title('+'.join(rad), fontsize=10, fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cocktailResponse_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Summary table

In [ ]:
print(f'{"Cocktail":<30} {"rho":>6}  {"Z/A":>6}  {"heavy":>6}  '
      f'{"H-3 eff_D":>11}  {"C-14 eff_D":>11}  {"Fe-55 eff_D":>12}')
print('─' * 100)

# Sort by Fe-55 eff_D
sorted_c = sorted(COCKTAILS, key=lambda c: results[c]['Fe-55']['eff_D'], reverse=True)

for name in sorted_c:
    p = props[name]
    dH  = results[name]['H-3']['eff_D'];  uH  = results[name]['H-3']['u_D']
    dC  = results[name]['C-14']['eff_D']; uC  = results[name]['C-14']['u_D']
    dFe = results[name]['Fe-55']['eff_D']; uFe = results[name]['Fe-55']['u_D']
    marker = ' ←' if name == REF else ''
    print(f'{name:<30} {p["rho"]:6.4f}  {p["Z"]/p["A"]:6.4f}  {p["heavy"]:6.3f}  '
          f'{dH:7.5f}±{uH:.5f}  {dC:7.5f}±{uC:.5f}  {dFe:8.5f}±{uFe:.5f}{marker}')